# Oracle Local Connection Check

Notebook simples para validar conexao com Oracle quando o projeto e o banco estao
na mesma maquina (ex.: docker-compose com network_mode: host).

Etapas:
- carrega ORACLE_* do .env (host/port default para localhost/1521)
- valida conexao TCP
- opcional: faz SELECT 1 FROM dual via Spark JDBC


In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "config").exists():
    project_root = cwd
elif (cwd.parent / "config").exists():
    project_root = cwd.parent
elif Path("/app").exists():
    project_root = Path("/app")
else:
    project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Local defaults when running on the same machine as the database.
os.environ.setdefault("ORACLE_HOST", "localhost")
os.environ.setdefault("ORACLE_PORT", "1521")

print("Checking Oracle connection settings...")

required = ["ORACLE_SERVICE", "ORACLE_USER", "ORACLE_PASSWORD"]
missing = [key for key in required if not os.getenv(key)]

oracle_config = None
if missing:
    print("Missing ORACLE_* values in .env or environment:")
    for key in missing:
        print(f"  - {key}")
    print("Fill them and re-run this cell.")
else:
    from config.settings import OracleSettings

    oracle_config = OracleSettings()
    display_config = {
        "host": oracle_config.host,
        "port": oracle_config.port,
        "service": oracle_config.service,
        "user": oracle_config.user,
    }
    display_config


Checking Oracle connection settings...


In [2]:
import socket

def check_tcp(host: str, port: int, timeout: float = 3.0) -> bool:
    sock = socket.socket()
    sock.settimeout(timeout)
    try:
        sock.connect((host, int(port)))
        print(f"TCP OK: {host}:{port}")
        return True
    except Exception as exc:
        print(f"TCP FAIL: {host}:{port} -> {exc}")
        return False
    finally:
        sock.close()

tcp_ok = False
if oracle_config:
    tcp_ok = check_tcp(oracle_config.host, oracle_config.port)
else:
    print("Skipping TCP check because Oracle config is missing.")


TCP FAIL: etbdb002:1522 -> [Errno -2] Name or service not known


In [3]:
import os
from pathlib import Path
from pyspark.sql import SparkSession

if not tcp_ok:
    print("Skipping Spark JDBC check (TCP not ok).")
else:
    spark_home = os.getenv("SPARK_HOME", "/usr/local/spark")
    ojdbc_jar = Path(spark_home) / "jars" / "ojdbc8-21.9.0.0.jar"

    if not ojdbc_jar.exists():
        print(f"ojdbc jar not found at: {ojdbc_jar}")
        print("Update SPARK_HOME or the jar path and re-run this cell.")
    else:
        spark = (
            SparkSession.builder.appName("oracle_local_connection_check")
            .config("spark.jars", str(ojdbc_jar))
            .getOrCreate()
        )

        oracle_jdbc_url = (
            f"jdbc:oracle:thin:@//{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"
        )

        df = (
            spark.read.format("jdbc")
            .option("url", oracle_jdbc_url)
            .option("dbtable", "(SELECT 1 AS ok FROM dual) t")
            .option("user", oracle_config.user)
            .option("password", oracle_config.password)
            .option("driver", "oracle.jdbc.driver.OracleDriver")
            .load()
        )
        df.show(truncate=False)


Skipping Spark JDBC check (TCP not ok).


In [ ]:
if "spark" in globals():
    spark.stop()
